# 🎬 Zonos2 — AI Video Dubbing
[GitHub: 5656wwed/SoniTranslate (zonos2 branch)](https://github.com/5656wwed/SoniTranslate/tree/zonos2)

**⚠️ Use T4 GPU** | **Accept pyannote license:** [hf.co/pyannote/speaker-diarization](https://hf.co/pyannote/speaker-diarization)
Get HF token: [hf.co/settings/tokens](https://hf.co/settings/tokens) (tick "Read access to public gated repos")

In [ ]:
#@markdown ## Step 1: Install (~15 min, includes Zonos 3GB model)

%cd /content
!rm -rf /content/SoniTranslate /content/Zonos
!git clone -q -b zonos2 https://github.com/5656wwed/SoniTranslate.git
%cd SoniTranslate

# ── Setup venv ──
!pip uninstall chex pandas-stubs ibis-framework albumentations albucore jax numpy -y -q 2>/dev/null
!pip install -q uv==0.8.13
!uv venv --python 3.10 --clear -q
!curl -sS https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!uv run python get-pip.py pip==23.1.2 -q
!uv run python -m pip install -q pip==23.1.2 Setuptools==80.6.0
!apt-get install -qq -y git-lfs ffmpeg espeak-ng 2>/dev/null
!git lfs install

# ── PyTorch ──
!uv run python -m pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

# ── Core deps ──
!sed -i 's|git+https://github.com/R3gm/whisperX.git@cuda_11_8|git+https://github.com/R3gm/whisperX.git@cuda_12_x|' requirements_base.txt
!uv run python -m pip install -q -r requirements_base.txt
!uv run python -m pip install -q -r requirements_extra.txt
!uv run python -m pip install -q onnxruntime-gpu==1.22.0

# ── TTS engines ──
!uv run python -m pip install -q piper-tts==1.2.0
!uv run python -m pip install -q -r requirements_xtts.txt 2>/dev/null
!uv run python -m pip install -q TTS==0.21.1 --no-deps 2>/dev/null
!uv run python -m pip install -q kokoro pocket-tts

# ── ZONOS ──
!git clone -q https://github.com/Zyphra/Zonos.git /content/Zonos
# Verify Zonos install
!uv run python -c "import zonos; print('Zonos version:', zonos.__version__ if hasattr(zonos, '__version__') else 'OK')" || echo "ZONOS INSTALL FAILED — check logs above"
# Install Zonos into the venv
!cd /content/Zonos && /content/SoniTranslate/.venv/bin/pip install -e .
%cd /content/Zonos
%cd /content/SoniTranslate

# ── numpy <2.0 (for pyannote) ──
!uv run python -m pip install -q "numpy<2.0" --force-reinstall --no-deps

# ── libcudnn8 (non-fatal) ──
!sudo apt-get install -y libcudnn8 -q 2>/dev/null || echo "libcudnn8 skipped"

print("\n✅ All done!")


In [ ]:
#@markdown ## Step 2: Mount Drive (for Zonos voice persistence)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/zonos_voices
print("✅ Drive mounted, voice storage ready")


In [ ]:
#@markdown ## Step 2.5: Upload & Process Zonos Voice
#@markdown Upload 5-30s WAV/MP3. Embedding saved to Drive.

from google.colab import files
uploaded = files.upload()
if uploaded:
    voice_path = list(uploaded.keys())[0]
    !cp "{voice_path}" /content/drive/MyDrive/zonos_voices/voice_sample.wav
    
    # Write then run Zonos embedding script
    with open('/tmp/zonos_embed.py', 'w') as f:
        f.write('''import torch\n'
                'torch.set_default_device("cuda")\n'
                'import torchaudio, sys\n'
                'from zonos.model import Zonos\n'
                'vp = sys.argv[1]\n'
                'model = Zonos.from_pretrained("Zyphra/Zonos-v0.1-transformer", device="cuda")\n'
                'wav, sr = torchaudio.load(vp)\n'
                'wav = wav.to("cuda")\n'
                'spk = model.make_speaker_embedding(wav, sr)\n'
                'torch.save(spk, "/content/drive/MyDrive/zonos_voices/default.pt")\n'
                'print("Embedding saved!")')
    !uv run python /tmp/zonos_embed.py "{voice_path}"
    %cd /content/SoniTranslate
    print(f"\n✅ Done! Pick 'en-Zonos-Default Zonos-TTS' from the dropdown.")


In [ ]:
#@markdown ## Step 3: 🚀 Launch

YOUR_HF_TOKEN = "" #@param {type:'string'}
%env YOUR_HF_TOKEN={YOUR_HF_TOKEN}

theme = "Taithrah/Minimal" # @param ["Taithrah/Minimal", "aliabid94/new-theme", "gstaff/xkcd", "gradio/dracula_revamped"]
interface_language = "english" # @param ['english', 'french', 'spanish', 'german', 'italian', 'portuguese', 'hindi', 'japanese', 'chinese_zh_cn', 'arabic', 'korean', 'russian']

%cd /content/SoniTranslate
# Skip WhisperX when using SRT — saves 3GB VRAM for Zonos
!uv run python app_rvc.py --theme {theme} --language {interface_language} --public_url
